# Token Classification (NER)

## Notebook Contract
- **Objective:** demonstrate the v0.7 token-classification workflow: a CoNLL-style NER data schema, subword alignment for HF tokenizers, an offline baseline trained with hand-crafted features, and entity-level BIO-span metrics.
- **Inputs:** synthetic NER data generated locally (no Hub downloads).
- **Outputs:** dataset preview, per-token feature inspection, alignment demo, entity-level report, and a CSV summary under `reports/sample_run/`.
- **Expected runtime:** under 30 seconds on CPU.
- **Scope boundary:** schema, alignment, and metrics live in `src/hf_finetuning_lab/token_classification/`; the notebook orchestrates and visualizes.

## 1) Setup and Reproducibility

In [1]:
from __future__ import annotations

import json
import os
import platform
import random
import sys
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path('..').resolve()
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

from hf_finetuning_lab.token_classification import (
    NERExample,
    align_word_labels_to_subwords,
    extract_entities,
    generate_synthetic_ner_data,
    sequence_tagging_report,
    validate_ner_dataset,
    write_synthetic_ner_jsonl,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f"Timestamp (UTC): {datetime.now(UTC).isoformat(timespec='seconds')}")
print(f'Seed: {SEED}')

Python: 3.13.5
Platform: Windows-11-10.0.26200-SP0
Timestamp (UTC): 2026-06-11T18:02:59+00:00
Seed: 42


## 2) Parameters

In [2]:
DATA_PATH = ROOT / 'data' / 'raw' / 'synthetic_ner.jsonl'
REPORTS_DIR = ROOT / 'reports' / 'sample_run'
NER_REPORT_PATH = REPORTS_DIR / 'ner_entity_report.csv'
NER_METRICS_PATH = REPORTS_DIR / 'ner_metrics.json'

N_EXAMPLES = 400
TEST_SIZE = 0.25
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Data path:   {DATA_PATH}')
print(f'Reports dir: {REPORTS_DIR}')

Data path:   C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\data\raw\synthetic_ner.jsonl
Reports dir: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run


## 3) Synthetic NER Data

The generator produces short CoNLL-style examples with `PER` / `ORG` / `LOC` entity types in BIO tagging scheme. Each line of the JSONL output is a single example with parallel `tokens` and `labels` arrays.

In [3]:
write_synthetic_ner_jsonl(DATA_PATH, n_examples=N_EXAMPLES, seed=SEED)
examples = generate_synthetic_ner_data(n_examples=N_EXAMPLES, seed=SEED)
validate_ner_dataset(examples)
print(f'Examples written to: {DATA_PATH}')
print(f'Loaded {len(examples)} examples')
first = examples[0]
preview = pd.DataFrame({'token': first.tokens, 'label': first.labels})
preview

Examples written to: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\data\raw\synthetic_ner.jsonl
Loaded 400 examples


,token,label
0,Liu,B-PER
1,Wei,I-PER
2,joined,O
3,Globex,B-ORG
4,Corp,I-ORG
5,in,O
6,Toronto,B-LOC
7,last,O
8,week,O
9,.,O


## 4) Label and Entity Distribution

In [4]:
label_counter: Counter[str] = Counter()
entity_counter: Counter[str] = Counter()
for example in examples:
    label_counter.update(example.labels)
    for entity_type, _, _ in extract_entities(example.labels):
        entity_counter[entity_type] += 1

label_share = (
    pd.Series(label_counter, name='count')
    .rename_axis('label')
    .reset_index()
    .sort_values('count', ascending=False)
)
label_share['share'] = (label_share['count'] / label_share['count'].sum()).round(4)

entity_share = (
    pd.Series(entity_counter, name='count')
    .rename_axis('entity_type')
    .reset_index()
    .sort_values('count', ascending=False)
)
entity_share['share'] = (entity_share['count'] / entity_share['count'].sum()).round(4)

print('BIO label distribution:')
print(label_share.to_string(index=False))
print('\nEntity span distribution:')
print(entity_share.to_string(index=False))

BIO label distribution:
label  count  share
    O   2269 0.5730
B-ORG    457 0.1154
B-LOC    343 0.0866
I-ORG    308 0.0778
B-PER    267 0.0674
I-PER    267 0.0674
I-LOC     49 0.0124

Entity span distribution:
entity_type  count  share
        ORG    457 0.4283
        LOC    343 0.3215
        PER    267 0.2502


## 5) Subword Alignment Demo

`align_word_labels_to_subwords` consumes the `word_ids` array produced by a Hugging Face fast tokenizer (`None` for special tokens, the source word index otherwise) and returns one entry per subword. Under the `first` strategy it keeps the original label on the first subword of each word and masks the continuation subwords (and special tokens) with `-100` — the standard Hugging Face convention, so the cross-entropy loss ignores those positions instead of being taught a contradictory `O` label. The `all` strategy instead propagates the label to every subword, rewriting `B-` to `I-` so the BIO scheme stays consistent. To stay offline we simulate a tokenizer with a tiny rule (`Lisbon` -> `Lis ##bon`) so the alignment logic can be inspected without a real model download.

In [5]:
def fake_subword_tokenize(words: list[str]) -> tuple[list[str], list[int | None]]:
    """Pretend to be an HF fast tokenizer: split long words after position 3."""
    sub_tokens: list[str] = ['[CLS]']
    word_ids: list[int | None] = [None]
    for idx, word in enumerate(words):
        if len(word) <= 3 or not word.isalpha():
            sub_tokens.append(word)
            word_ids.append(idx)
        else:
            sub_tokens.append(word[:3])
            word_ids.append(idx)
            sub_tokens.append('##' + word[3:])
            word_ids.append(idx)
    sub_tokens.append('[SEP]')
    word_ids.append(None)
    return sub_tokens, word_ids


sample = examples[0]
sub_tokens, word_ids = fake_subword_tokenize(sample.tokens)
first_aligned = align_word_labels_to_subwords(sample.labels, word_ids, strategy='first')
all_aligned = align_word_labels_to_subwords(sample.labels, word_ids, strategy='all')

alignment_preview = pd.DataFrame(
    {
        'subword_token': sub_tokens,
        'word_id': word_ids,
        'aligned_first': first_aligned,
        'aligned_all': all_aligned,
    }
)
alignment_preview

,subword_token,word_id,aligned_first,aligned_all
0,[CLS],NaN,-100,-100
1,Liu,0.0,B-PER,B-PER
2,Wei,1.0,I-PER,I-PER
3,joi,2.0,O,O
4,##ned,2.0,-100,O
5,Glo,3.0,B-ORG,B-ORG
6,##bex,3.0,-100,I-ORG
7,Cor,4.0,I-ORG,I-ORG
8,##p,4.0,-100,I-ORG
9,in,5.0,O,O


## 6) Per-Token Features for the Baseline

A small set of hand-crafted features lets a per-token logistic regression learn the BIO scheme on this synthetic data. With a real NER dataset the same workflow plugs into a transformer; here we keep it CPU-friendly.

In [6]:
def token_features(tokens: list[str], position: int) -> dict[str, str | bool]:
    token = tokens[position]
    return {
        'token_lower': token.lower(),
        'prefix3': token[:3].lower(),
        'suffix3': token[-3:].lower(),
        'is_title': token.istitle(),
        'is_upper': token.isupper(),
        'is_first': position == 0,
        'has_digit': any(ch.isdigit() for ch in token),
        'prev_token': tokens[position - 1].lower() if position > 0 else '<BOS>',
        'next_token': tokens[position + 1].lower() if position + 1 < len(tokens) else '<EOS>',
    }


def build_dataset(examples: list[NERExample]) -> tuple[list[dict[str, str | bool]], list[str]]:
    feats: list[dict[str, str | bool]] = []
    labels: list[str] = []
    for example in examples:
        for i in range(len(example.tokens)):
            feats.append(token_features(example.tokens, i))
            labels.append(example.labels[i])
    return feats, labels


rng = np.random.default_rng(SEED)
shuffle = rng.permutation(len(examples))
split_at = int(len(examples) * (1 - TEST_SIZE))
train_idx, test_idx = shuffle[:split_at], shuffle[split_at:]
train_examples = [examples[i] for i in train_idx]
test_examples = [examples[i] for i in test_idx]

train_feats, train_labels = build_dataset(train_examples)
test_feats, test_labels = build_dataset(test_examples)
print(f'Train tokens: {len(train_labels)}')
print(f'Test tokens:  {len(test_labels)}')
pd.DataFrame([token_features(examples[0].tokens, i) for i in range(min(6, len(examples[0].tokens)))])

Train tokens: 2957
Test tokens:  1003


,token_lower,prefix3,suffix3,is_title,is_upper,is_first,has_digit,prev_token,next_token
0,liu,liu,liu,True,False,True,False,<BOS>,wei
1,wei,wei,wei,True,False,False,False,liu,joined
2,joined,joi,ned,False,False,False,False,wei,globex
3,globex,glo,bex,True,False,False,False,joined,corp
4,corp,cor,orp,True,False,False,False,globex,in
5,in,in,in,False,False,False,False,corp,toronto


## 7) Train a Per-Token Logistic Regression

In [7]:
vectorizer = DictVectorizer(sparse=True)
X_train = vectorizer.fit_transform(train_feats)
X_test = vectorizer.transform(test_feats)

clf = LogisticRegression(max_iter=2000, C=4.0, random_state=SEED)
clf.fit(X_train, train_labels)
pred_labels = clf.predict(X_test)

# Reshape the flat prediction list back into per-example sequences.
pred_sequences: list[list[str]] = []
cursor = 0
for example in test_examples:
    n = len(example.tokens)
    pred_sequences.append(list(pred_labels[cursor : cursor + n]))
    cursor += n
true_sequences = [example.labels for example in test_examples]

token_accuracy = float((np.array(pred_labels) == np.array(test_labels)).mean())
print(f'Per-token accuracy: {token_accuracy:.4f}')
print('First test example:')
pd.DataFrame(
    {
        'token': test_examples[0].tokens,
        'true': test_examples[0].labels,
        'pred': pred_sequences[0],
    }
)

Per-token accuracy: 1.0000
First test example:


,token,true,pred
0,The,O,O
1,Pied,B-ORG,B-ORG
2,Piper,I-ORG,I-ORG
3,headquarters,O,O
4,moved,O,O
5,to,O,O
6,Toronto,B-LOC,B-LOC
7,last,O,O
8,year,O,O
9,.,O,O


## 8) Entity-Level Report

Per-token accuracy can be misleading because most tokens are `O`. The entity-level report counts span hits (matching type and boundaries) — the metric that matters for downstream IE applications.

In [8]:
report = sequence_tagging_report(true_sequences, pred_sequences).round(4)
report.to_csv(NER_REPORT_PATH)
print(f'Report written to: {NER_REPORT_PATH}')
report

Report written to: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\ner_entity_report.csv


,precision,recall,f1,support
entity_type,,,,
LOC,1.0,1.0,1.0,88
ORG,1.0,1.0,1.0,112
PER,1.0,1.0,1.0,71
micro avg,1.0,1.0,1.0,271
macro avg,1.0,1.0,1.0,271


In [9]:
summary = {
    'n_train_examples': int(len(train_examples)),
    'n_test_examples': int(len(test_examples)),
    'per_token_accuracy': token_accuracy,
    'entity_micro_f1': float(report.loc['micro avg', 'f1']),
    'entity_macro_f1': float(report.loc['macro avg', 'f1']),
}
NER_METRICS_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f'Summary written to: {NER_METRICS_PATH}')
summary

Summary written to: C:\Users\diogo\work_code\portfolio\huggingface-finetuning-lab\reports\sample_run\ner_metrics.json


{'n_train_examples': 300,
 'n_test_examples': 100,
 'per_token_accuracy': 1.0,
 'entity_micro_f1': 1.0,
 'entity_macro_f1': 1.0}

## 9) Error Inspection

Sample a few test examples where the predicted entity set differs from the gold set. These are the rows to look at when iterating on features or moving to a transformer.

In [10]:
mismatches: list[dict[str, object]] = []
for example, pred in zip(test_examples, pred_sequences, strict=True):
    true_spans = set(extract_entities(example.labels))
    pred_spans = set(extract_entities(pred))
    if true_spans != pred_spans:
        mismatches.append(
            {
                'text': ' '.join(example.tokens),
                'gold': sorted(true_spans),
                'pred': sorted(pred_spans),
            }
        )
        if len(mismatches) >= 5:
            break
pd.DataFrame(mismatches)

""


## 10) Checklist
- NER schema (`NERExample`) plus generator/validator/writer keeps token-classification data lifecycle in one module.
- Subword alignment is library-agnostic: it operates on the `word_ids` array that any HF fast tokenizer can produce, so the same function fits into a transformer fine-tune.
- Entity-level micro/macro F1 (`sequence_tagging_report`) is the right metric for span-based evaluation; per-token accuracy alone is misleading.
- For a production run: replace the synthetic generator with a real CoNLL-style corpus, swap the per-token baseline for a transformer fine-tune, and add the v0.4 robust-evaluation notebook's bootstrap CIs to the report.